# Setup

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import os
import sys
import pandas as pd

# enforce more deterministic behavior in cuBLAS operations.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# select a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sys.path.append("..")

from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import AbstractDocument
from processor.core.ir_system.ir_data_model import Table, TableContext
from processor.core.ir_system.ir_data_model import Knowledge, convert_multi_retriever_results_to_str
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.model.llm_message import Role
from processor.model.option import LLMOption
from processor.utils.json_processor import parse_json

from logging import Logger
from pandas import DataFrame

logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)

# Dev

## Impl

In [2]:
import json

from pandas import DataFrame

from processor.core.ir_system.ir_data_model import (
    AbstractDocument,
)


class MEPromptFactory:
    def get_planning_prompt(
        self,
        target_schemas: dict[str, DataFrame],
        column_descriptions: dict[str, dict[str, str]],
        sqls: list[str],
        tool_description: str,
        operation_description: str,
    ) -> str:
        return f"""You are a smart data scientist planning to materialize a set of table schemas that we refer to as Target Schemas:
```{json.dumps({k: list(df.columns) for k, df in target_schemas.items()}, indent=2)}```

This is the descriptions of the columns in Target Schemas:
```{column_descriptions}```

Also, just for reference (you will not need to execute this), these are the SQL queries that will be executed sequentially over the final target tables, which are materialized Target Schemas (observe the expected value format in the queries):
```{sqls}```

Again, your goal is to actually materialize the Target Schemas by manipulating tables/textual information in our database (retrieved from the Document Retriever tool).
You can select, integrate (join, union), transform values of, etc. these retrieved tables using the available tools and operations.

Available Tools:
```{tool_description}```

Available Operations:
```{operation_description}```

IMPORTANT: There are no other tools and operations, so use ONLY the above tools and operations."""

    def get_context_prompt(
        self,
        retrieved_documents: dict[RetrieverType, list[AbstractDocument]],
        intermediate_tables: dict[str, DataFrame],
        recent_actions: list[str],
        num_iterations: int,
    ) -> str:
        return f"""So far, we have reacted {num_iterations} times to the instructions. Below is our progress:

- Current intermediate tables (what we have formed so far, again we want to make sure to finally materialize Target Schemas into this list): ```{list(intermediate_tables.keys())}```

- Most recently taken actions: ```{recent_actions}```

- The documents we previously retrieved (tables and/or textual information): ```{convert_multi_retriever_results_to_str(retrieved_documents)}```

IMPORTANT: Before retrieving new documents, check if existing documents above contain the information we need. Only retrieve new documents if the current ones do not have what we are looking for.

Plan our next step using this format:
{{
  "step_type": "operation" | "tool" | "internal_reasoning",
  "message": null (if step_type is "operation" or "tool") | <"Reflect out loud (for ourselves only)">
  "name": null (if step_type is "internal_reasoning") | "Document Retriever" | "Python Executor" | "SQL Executor" | "Standard Inner Join" | "Union",
  "args": null (if step_type is "internal_reasoning") | {{"The argument to the function or tool that we call"}}
  "assign_to": null (if step_type is "internal_reasoning") | "result_table_id"  # Must match one of the target schema IDs if this is a final result (i.e., correspond to a target schema directly)
}}"""

In [3]:
from typing import Any, Optional

from logging import Logger
from pandas import DataFrame
from processor.core.materializer_engine.me_state import MaterializerState
from processor.core.materializer_engine.operation.operation_description import (
    get_operation_description,
)
from processor.core.materializer_engine.operation.std_inner_join import std_inner_join
from processor.core.materializer_engine.operation.union import union
from processor.core.materializer_engine.tool.document_retriever import get_documents
from processor.core.materializer_engine.tool.python_executor import execute_python_code
from processor.core.materializer_engine.tool.sql_executor import execute_sql
from processor.core.materializer_engine.tool.tool_description import (
    get_tool_description,
)
from processor.model.interface.abstract_model import AbstractModel
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption
from processor.utils.json_processor import parse_json


class LLMPlanner:
    def __init__(self, llm: AbstractModel, logger: Logger, embed_model: AbstractModel):
        self.logger = logger
        self.logger.info(
            "Initializing LLMPlanner, the core component of Materializer Engine"
        )
        self.llm = llm
        self.embed_model = embed_model

        self.prompt_factory = MEPromptFactory()
        self.state = MaterializerState()

        self.actions: list[str] = []

    def materialize_target_schemas(
        self,
        target_schemas: dict[str, DataFrame],
        column_descriptions: dict[str, dict[str, str]],
        sqls: list[str],
        feedback: Optional[str] = None,
    ) -> dict[str, DataFrame]:
        self.logger.info(
            f"Starting materialization for {len(target_schemas)} target schemas with {len(sqls)} SQLs"
        )
        self.state.reset()
        sys_prompt = LLMMessage(
            role=Role.SYSTEM.value,
            content=self.prompt_factory.get_planning_prompt(
                target_schemas=target_schemas,
                column_descriptions=column_descriptions,
                sqls=sqls,
                tool_description=get_tool_description(),
                operation_description=get_operation_description(),
            ),
        )
        num_iterations = 0
        llm_messages = [sys_prompt]
        while not self.__check_completion(target_schemas):
            self.logger.info("Planning next materialization step")
            self.logger.info("Requesting LLM response for plan")
            llm_messages.append(
                LLMMessage(
                    role=Role.USER.value,
                    content=self.prompt_factory.get_context_prompt(
                        self.state.current_retrieved_documents,
                        self.state.intermediate_tables,
                        self.actions,
                        num_iterations,
                    ),
                )
            )

            num_iterations += 1
            response = self.llm.chat(llm_messages, LLMOption(json_mode=True))
            llm_messages.append(
                LLMMessage(
                    role=Role.ASSISTANT.value,
                    content=response,
                )
            )
            self.logger.info(f"LLM response: {response}")
            """
            Output format:
            {{
                "step_type": "operation" | "tool" | "internal_reasoning",
                "message": null (if step_type is "operation" or "tool") | <"Reflect out loud (for yourself only)">
                "name": null (if step_type is "internal_reasoning") | "Operation or Tool name",
                "args": null (if step_type is "internal_reasoning") | {{"The argument to the function or tool that you call"}}
                "assign_to": null (if step_type is "internal_reasoning") | "result_table_id"  # Must match one of the target schema IDs if this is a final result
            }}
            """
            plan: dict[str, Any] = parse_json(response)
            step_type: str = plan["step_type"]

            curr_retrieved_docs_tables_only: dict[str, DataFrame] = dict()
            for retriever_type in self.state.current_retrieved_documents:
                if retriever_type == RetrieverType.PNEUMA:
                    docs = self.state.current_retrieved_documents[retriever_type]
                    for doc in docs:
                        curr_retrieved_docs_tables_only[doc.doc_id] = doc.content
            all_tables = {
                **curr_retrieved_docs_tables_only,
                **self.state.intermediate_tables,
            }

            if step_type == "internal_reasoning":
                message: str = plan["message"]
                self.actions.append(f"Reasoned internally: {message}")
            elif step_type == "operation" or step_type == "tool":
                op_name: str = plan["name"]
                op_args: dict[str, Any] = plan["args"]
                assign_to: str = plan["assign_to"]
                if op_name == "Standard Inner Join":
                    left_table_id: str = op_args["left_table_id"]
                    right_table_id: str = op_args["right_table_id"]
                    join_key: str = op_args["join_key"]
                    join_res = std_inner_join(
                        left_table_id,
                        right_table_id,
                        all_tables,
                        join_key,
                    )
                    self.state.intermediate_tables[assign_to] = join_res
                    self.actions.append(
                        f"Performed standard inner join between {left_table_id} and {right_table_id} with join key {join_key}, resulting in {assign_to}"
                    )
                elif op_name == "Union":
                    table_ids: list[str] = op_args["table_ids"]
                    union_res = union(all_tables, table_ids)
                    self.state.intermediate_tables[assign_to] = union_res
                    self.actions.append(
                        f"Performed union between these tables: {table_ids}, resulting in {assign_to}"
                    )
                elif op_name == "Document Retriever":
                    prompt: str = op_args["prompt"]
                    self.state.current_retrieved_documents = get_documents(
                        self.llm, self.embed_model, self.logger, prompt, ["environment"]
                    )
                    self.actions.append(
                        f'Successfully retrieved documents using this prompt: ```{prompt}```. Notice that the "Previously retrieved documents" have been filled.'
                    )
                elif op_name == "Python Executor":
                    python_code: str = op_args["code"]
                    exec_res = execute_python_code(python_code, all_tables, self.logger)
                    if isinstance(exec_res, DataFrame):
                        self.state.intermediate_tables[assign_to] = exec_res
                        self.actions.append(
                            f"Successfully executed the Python code, resulting in a table named {assign_to}"
                        )
                    else:
                        self.actions.append(
                            f"Successfully executed the Python code, resulting in this: {exec_res}"
                        )
                elif op_name == "SQL Executor":
                    sql_query: str = op_args["sql_query"]
                    exec_res = execute_sql(self.logger, sql_query, all_tables)
                    self.state.intermediate_tables[assign_to] = exec_res
                    self.actions.append(
                        f"Successfully executed the SQL query, resulting in a table named {assign_to}"
                    )
                else:
                    self.actions.append(
                        f"Trying to perform/execute {op_name}, but it is not a valid operation."
                    )
            else:
                self.actions.append(f"The step {step_type} is not a valid action.")
        self.logger.info("Materialization completed successfully")
        final_result: dict[str, DataFrame] = dict()
        for key, value in self.state.intermediate_tables.items():
            if key in target_schemas.keys():
                final_result[key] = value
        return final_result

    def __check_completion(self, target_schemas: dict[str, DataFrame]) -> bool:
        print(f"DEBUGGY: CHECK COMPLETION")
        all_schema_ids = set(target_schemas.keys())
        materialized_schema_ids = set(self.state.intermediate_tables.keys())
        print(f"==> all_schema_ids: {all_schema_ids}")
        print(f"==> materialized_schema_ids: {materialized_schema_ids}")

        is_complete = all_schema_ids <= materialized_schema_ids

        print(f"==> is_complete: {is_complete}")

        self.logger.info(
            f"Completion check: {is_complete} ({len(materialized_schema_ids)}/{len(all_schema_ids)} schemas materialized)"
        )
        return is_complete

## Testing

In [4]:
llm_path = "model/weight/qwen3-8b"
llm = get_llm(llm_path)(llm_path)
embed_model_path = "model/weight/bge-base"
embed_model = get_embed_model()(embed_model_path)

In [5]:
materializer_engine = LLMPlanner(
    llm=llm,
    logger=logger,
    embed_model=embed_model,
)

[2025-07-27 11:33:29] INFO in 1342225053: Initializing LLMPlanner, the core component of Materializer Engine


In [6]:
target_schemas: dict[str, DataFrame] = {
    "malibu_beach_testing": DataFrame(columns=["Date","1-Day Rain","2-Day Rain","3-Day Rain","Tag","Enterococcus"])
}
column_descriptions: dict[str, dict[str, str]] = {
    "malibu_beach_testing": {
        "Date": "The date",
        "1-Day Rain": "Total rain day 1",
        "2-Day Rain": "Total rain day 1-2",
        "3-Day Rain": "Total rain day 1-3",
        "Tag": "Tag measurement",
        "Enterococcus": "Enterococcus measurement"
    }
}
sqls: list[str] = [
    "SELECT * FROM malibu_beach_testing WHERE Date = `2024-08-31`"
]

In [7]:
materialized_target_schemas = materializer_engine.materialize_target_schemas(
    target_schemas, column_descriptions, sqls
)
materialized_target_schemas

[2025-07-27 11:33:29] INFO in 1342225053: Starting materialization for 1 target schemas with 1 SQLs
DEBUGGY: CHECK COMPLETION
==> all_schema_ids: {'malibu_beach_testing'}
==> materialized_schema_ids: set()
==> is_complete: False
[2025-07-27 11:33:29] INFO in 1342225053: Completion check: False (0/1 schemas materialized)
[2025-07-27 11:33:29] INFO in 1342225053: Planning next materialization step
[2025-07-27 11:33:29] INFO in 1342225053: Requesting LLM response for plan


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

QWEN: response: {
  "step_type": "tool",
  "name": "Document Retriever",
  "args": {
    "prompt": "Get data for malibu_beach_testing including Date, 1-Day Rain, 2-Day Rain, 3-Day Rain, Tag, and Enterococcus measurements"
  },
  "assign_to": "malibu_beach_testing"
}
[2025-07-27 11:34:05] INFO in 1342225053: LLM response: {
  "step_type": "tool",
  "name": "Document Retriever",
  "args": {
    "prompt": "Get data for malibu_beach_testing including Date, 1-Day Rain, 2-Day Rain, 3-Day Rain, Tag, and Enterococcus measurements"
  },
  "assign_to": "malibu_beach_testing"
}
[2025-07-27 11:34:05] INFO in lm_interface: Starting document retrieval for prompt: Get data for malibu_beach_testing including Date, 1-Day Rain, 2-Day Rain, 3-Day Rain, Tag, and Enter...
[2025-07-27 11:34:05] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>, <RetrieverType.KNOWLEDGE_BASE: 'Knowledge Base'>]
[2025-07-27 11:34:05] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA
[20

{'malibu_beach_testing':          Date  1-Day Rain  2-Day Rain  3-Day Rain   Tag  Enterococcus
 4  2024-08-28        0.00        0.00        0.00  <NA>            10
 5  2024-08-27        0.00        0.00        0.00  <NA>            10
 6  2024-08-26        0.00        0.00        0.00  <NA>            10
 8  2024-08-24        0.00        0.00        0.00  <NA>            10
 18 2024-08-14        0.00        0.00        0.00  <NA>            10
 19 2024-08-13        0.00        0.00        0.00  <NA>            10
 20 2024-08-12        0.00        0.00        0.11  <NA>            10
 32 2024-07-31        0.20        0.20        0.23  <NA>            10
 36 2024-07-27        0.00        0.00        0.00  <NA>            10
 53 2024-07-10        0.00        0.00        0.00  <NA>            10
 55 2024-07-08        0.00        0.00        0.00  <NA>            10
 62 2024-07-01        0.16        0.18        0.18  <NA>            10
 63 2024-06-30        0.02        0.02        0.02  <

In [12]:
sqls[0]

'SELECT * FROM malibu_beach_testing WHERE Date = `2024-08-31`'

In [11]:
materialized_target_schemas['malibu_beach_testing']

,Date,1-Day Rain,2-Day Rain,3-Day Rain,Tag,Enterococcus
4,2024-08-28,0.00,0.00,0.00,<NA>,10
5,2024-08-27,0.00,0.00,0.00,<NA>,10
6,2024-08-26,0.00,0.00,0.00,<NA>,10
8,2024-08-24,0.00,0.00,0.00,<NA>,10
18,2024-08-14,0.00,0.00,0.00,<NA>,10
19,2024-08-13,0.00,0.00,0.00,<NA>,10
20,2024-08-12,0.00,0.00,0.11,<NA>,10
32,2024-07-31,0.20,0.20,0.23,<NA>,10
36,2024-07-27,0.00,0.00,0.00,<NA>,10
53,2024-07-10,0.00,0.00,0.00,<NA>,10
